# Cross-Validation

**Project question:** How can I compare model complexity without relying on one fortunate split?

By the end of this notebook, you should be able to:

- run reproducible K-fold cross-validation
- summarize fold-to-fold uncertainty rather than only the mean
- select complexity without treating CV as a final untouched test

This notebook is a demonstration, not a homework assignment. The data are
synthetic and were generated for teaching; numerical results should not be
interpreted as evidence about a real organization or population.

In [ ]:

from lite_setup import ensure_packages
await ensure_packages()

In [ ]:

import math
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

DATA = Path("data")
if not DATA.exists():
    raise FileNotFoundError("Open this notebook from the JupyterLite files root so data/ is available.")

In [ ]:
from sklearn.model_selection import KFold, cross_val_score
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import make_pipeline

In [ ]:
df = pd.read_csv(DATA / 'evaluation_regression.csv')
X = df[['x1', 'x2', 'x3']]
y = df['y']
cv = KFold(n_splits=5, shuffle=True, random_state=4031)

In [ ]:
rows = []
for degree in range(1, 6):
    model = make_pipeline(PolynomialFeatures(degree=degree, include_bias=False), LinearRegression())
    scores = -cross_val_score(model, X, y, scoring='neg_root_mean_squared_error', cv=cv)
    rows.append({
        'degree': degree,
        'cv_rmse_mean': scores.mean(),
        'cv_rmse_sd': scores.std(ddof=1),
        'cv_rmse_se': scores.std(ddof=1) / np.sqrt(len(scores)),
    })
cv_results = pd.DataFrame(rows)
cv_results

In [ ]:
best_degree = int(cv_results.loc[cv_results['cv_rmse_mean'].idxmin(), 'degree'])
plt.errorbar(
    cv_results['degree'], cv_results['cv_rmse_mean'],
    yerr=cv_results['cv_rmse_se'], marker='o', capsize=4
)
plt.axvline(best_degree, color='gray', linestyle='--', label=f'minimum mean: degree {best_degree}')
plt.xlabel('Polynomial degree')
plt.ylabel('5-fold CV RMSE')
plt.title('Cross-validation error vs model complexity')
plt.legend()

**Interpretation:** The standard error bars show that small rank differences may be sampling noise. Prefer the simpler model when its performance is practically indistinguishable. Because the same CV results chose the degree and estimated its performance, use an outer test set or nested CV for a final unbiased estimate.

**Transfer exercise:** Predeclare a metric and fold design for your project. Explain whether rows are independent enough for random K-fold CV or require grouped, blocked, or time-ordered splits.